In [ ]:
import numpy as np
import pandas as pd


df = pd.read_csv("cs-training.csv")

df.head()

,Unnamed: 0,SeriousDlqin2yrs,RevolvingUtilizationOfUnsecuredLines,age,NumberOfTime30-59DaysPastDueNotWorse,DebtRatio,MonthlyIncome,NumberOfOpenCreditLinesAndLoans,NumberOfTimes90DaysLate,NumberRealEstateLoansOrLines,NumberOfTime60-89DaysPastDueNotWorse,NumberOfDependents
0,1,1,0.766127,45,2,0.802982,9120.0,13,0,6,0,2.0
1,2,0,0.957151,40,0,0.121876,2600.0,4,0,0,0,1.0
2,3,0,0.658180,38,1,0.085113,3042.0,2,1,0,0,0.0
3,4,0,0.233810,30,0,0.036050,3300.0,5,0,0,0,0.0
4,5,0,0.907239,49,1,0.024926,63588.0,7,0,1,0,0.0


In [ ]:
df = df.drop(columns=["Unnamed: 0"])

In [ ]:
#verificar si existen valores NaN en las columnas
df.isna().sum()

,0
SeriousDlqin2yrs,0
RevolvingUtilizationOfUnsecuredLines,0
age,0
NumberOfTime30-59DaysPastDueNotWorse,0
DebtRatio,0
MonthlyIncome,29731
NumberOfOpenCreditLinesAndLoans,0
NumberOfTimes90DaysLate,0
NumberRealEstateLoansOrLines,0
NumberOfTime60-89DaysPastDueNotWorse,0


In [ ]:
#rellenar los nan
df["MonthlyIncome"] = df["MonthlyIncome"].fillna(df["MonthlyIncome"].median())
df["NumberOfDependents"] = df["NumberOfDependents"].fillna(df["NumberOfDependents"].median())

In [ ]:
X = df.drop(columns=["SeriousDlqin2yrs"])
y = df["SeriousDlqin2yrs"]

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [ ]:
from sklearn.preprocessing  import StandardScaler
#escalado solo para logistic regresion
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

log_model = LogisticRegression(max_iter=1000)

log_model.fit(X_train_scaled, y_train)

y_train_pred = log_model.predict(X_train_scaled)
y_test_pred = log_model.predict(X_test_scaled)

log_train_acc = accuracy_score(y_train, y_train_pred)
log_test_acc = accuracy_score(y_test, y_test_pred)

print("Logistic Train Accuracy:", log_train_acc)
print("Logistic Test Accuracy:", log_test_acc)

Logistic Train Accuracy: 0.9337083333333334
Logistic Test Accuracy: 0.9350333333333334


In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_unlimited = DecisionTreeClassifier(random_state=42)

tree_unlimited.fit(X_train, y_train)

train_pred = tree_unlimited.predict(X_train)
test_pred = tree_unlimited.predict(X_test)

tree_train_acc = accuracy_score(y_train, train_pred)
tree_test_acc = accuracy_score(y_test, test_pred)

print("Tree Unlimited Train:", tree_train_acc)
print("Tree Unlimited Test:", tree_test_acc)

Tree Unlimited Train: 0.9996416666666667
Tree Unlimited Test: 0.8968


In [ ]:
tree_limited = DecisionTreeClassifier(
    max_depth=5,
    random_state=42
)

tree_limited.fit(X_train, y_train)

train_pred = tree_limited.predict(X_train)
test_pred = tree_limited.predict(X_test)

tree_lim_train_acc = accuracy_score(y_train, train_pred)
tree_lim_test_acc = accuracy_score(y_test, test_pred)

print("Tree Limited Train:", tree_lim_train_acc)
print("Tree Limited Test:", tree_lim_test_acc)

Tree Limited Train: 0.9367333333333333
Tree Limited Test: 0.9363


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf_model.fit(X_train, y_train)

train_pred = rf_model.predict(X_train)
test_pred = rf_model.predict(X_test)

rf_train_acc = accuracy_score(y_train, train_pred)
rf_test_acc = accuracy_score(y_test, test_pred)

print("Random Forest Train:", rf_train_acc)
print("Random Forest Test:", rf_test_acc)

Random Forest Train: 0.9996416666666667
Random Forest Test: 0.9374


In [ ]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree Unlimited",
        "Decision Tree Limited",
        "Random Forest"
    ],
    "Train Accuracy": [
        log_train_acc,
        tree_train_acc,
        tree_lim_train_acc,
        rf_train_acc
    ],
    "Test Accuracy": [
        log_test_acc,
        tree_test_acc,
        tree_lim_test_acc,
        rf_test_acc
    ]
})

results

,Model,Train Accuracy,Test Accuracy
0,Logistic Regression,0.933708,0.935033
1,Decision Tree Unlimited,0.999642,0.896800
2,Decision Tree Limited,0.936733,0.936300
3,Random Forest,0.999642,0.937400


In [ ]:
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)
y_prob = rf_model.predict_proba(X_test)[:,1]

print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

Precision: 0.5544692737430168
Recall: 0.20296523517382414
F1: 0.2971556886227545
ROC AUC: 0.8423425928842757


In [ ]:
import pandas as pd

feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print(feature_importance)

RevolvingUtilizationOfUnsecuredLines    0.190842
DebtRatio                               0.178633
MonthlyIncome                           0.146128
age                                     0.127474
NumberOfTimes90DaysLate                 0.095939
NumberOfOpenCreditLinesAndLoans         0.088439
NumberOfTime30-59DaysPastDueNotWorse    0.048957
NumberOfTime60-89DaysPastDueNotWorse    0.047241
NumberOfDependents                      0.042107
NumberRealEstateLoansOrLines            0.034239
dtype: float64


In [ ]:
final_model = rf_model
final_model.fit(X, y)

RandomForestClassifier(n_estimators=200, random_state=42)

In [ ]:
test_data = pd.read_csv("cs-test.csv")

ids = test_data["Unnamed: 0"]

test_data = test_data.drop(["Unnamed: 0", "SeriousDlqin2yrs"], axis=1)

predictions = final_model.predict_proba(test_data)[:,1]

submission = pd.DataFrame({
    "Id": ids,
    "Probability": predictions
})

submission.to_csv("submission.csv", index=False)